In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from filters.classifier_based import TABPFNClassificationFilter
def add_class_noise(y, noise_rate=0.7, random_state=42):
    rng = np.random.default_rng(random_state)
    y_noisy = y.copy()
    classes = np.unique(y)
    n_flip = int(len(y) * noise_rate)
    flip_idx = rng.choice(len(y), size=n_flip, replace=False)
    for i in flip_idx:
        other = classes[classes != y_noisy[i]]
        y_noisy[i] = rng.choice(other)
    return y_noisy
# dataset
X, y = load_breast_cancer(return_X_y=True)
y_noisy = add_class_noise(y, noise_rate=0.25, random_state=42)
# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_noisy, test_size=0.3, random_state=42, stratify=y_noisy
)
# baseline
clf = LogisticRegression(max_iter=5000, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print("Baseline")
print("accuracy:", accuracy_score(y_test, y_pred))
print("f1:", f1_score(y_test, y_pred, average="weighted"))
print("balanced_accuracy:", balanced_accuracy_score(y_test, y_pred))
# filtro TABPFN
filt = TABPFNClassificationFilter(cv=5, action="remove", random_state=33)
X_train_clean, y_train_clean = filt.fit_resample(X_train, y_train)
print("\nTABPFN report")
print(filt.get_filter_report())
# modelo tras filtrar
clf_clean = LogisticRegression(max_iter=5000, random_state=42)
clf_clean.fit(X_train_clean, y_train_clean)
y_pred_clean = clf_clean.predict(X_test)
print("\nAfter TABPFN")
print("accuracy:", accuracy_score(y_test, y_pred_clean))
print("f1:", f1_score(y_test, y_pred_clean, average="weighted"))
print("balanced_accuracy:", balanced_accuracy_score(y_test, y_pred_clean))

/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Baseline
accuracy: 0.7368421052631579
f1: 0.730151888533868
balanced_accuracy: 0.7259218492019812


/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/tabpfn/validation.py:139: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(
/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/tabpfn/validation.py:139: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(
/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/tabpfn/validation.py:139: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(
/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/tabpfn/validation.py:139: UserWarning: Running on CPU with more than 200 samples may be slow.


TABPFN report
{'n_samples': 398, 'removed_or_flagged': 124, 'fraction_flagged': 0.31155778894472363, 'cv': 5, 'action': 'remove', 'tabpfn_params': {}}

After TABPFN
accuracy: 0.7543859649122807
f1: 0.7518110344354035
balanced_accuracy: 0.7475921849201981
